In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, time, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [3]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = 'USR_GSIT_GCHIARELLA'
pw = 'JghcnTr46W$aHcS'
hostname='172.20.0.109'
service_name="ospeprd"
port = 1521

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [4]:
QUERY1 = """select 
  count(SU.ID),
  to_char(sr.fecha_registro,'mm/yyyy')
from APP_ospe_virtual.segusuario su
join APP_ospe_virtual.segciudadano sc on SC.id= su.id_ciudadano
join APP_ospe_virtual.segusuario_rol sr on sr.id_usuario=su.id
left JOIN app_ospe_virtual.ospedependencia od on od.id =sr.id_dependencia_ospe
where 
    sr.estado in ('1','2') --0:inactivo,1:activo,2:por confirmar
AND  sr.ID_ROL in ('1') --1:asegurado, 2:empleador,3,operador_ospe,4:operador_ti,5:operador_bo
GROUP BY 
to_char(sr.fecha_registro,'mm/yyyy')
"""


QUERY2 = """select 
  count(SU.ID),
  to_char(sr.fecha_registro,'mm/yyyy')
from APP_ospe_virtual.segusuario su
join APP_ospe_virtual.segciudadano sc on SC.id= su.id_ciudadano
join APP_ospe_virtual.segusuario_rol sr on sr.id_usuario=su.id
where 
    sr.estado in ('1','2') --0:inactivo,1:activo,2:por confirmar
AND sr.ID_ROL in ('2') --1:asegurado, 2:empleador,3,operador_ospe,4:operador_ti,5:operador_bo
GROUP BY 
to_char(sr.fecha_registro,'mm/yyyy')"""

In [5]:
usuarios = pd.read_sql_query(QUERY1, con=connection0)
empresas = pd.read_sql_query(QUERY2, con=connection0)

connection0.close()
engine0.dispose()

usuarios.to_sql(name=f'cantidad_usuarios_viva', con=engine2, if_exists='replace', index=False)
empresas.to_sql(name=f'cantidad_empresas_viva', con=engine2, if_exists='replace', index=False)

60